# 🔥 Burn Severity — Benchmark 2: Pipeline vs Standalone
### YOLOv8x-seg · leak-free · runs on H100

Trains two segmentation models on the **identical** source-grouped, leak-free split (seed 42, 205-image test set) used by the classifier benchmark:

1. **1-class localiser** — the segmentation stage of the deployed pipeline (finds the burn region → mask).
2. **3-class standalone** — a segment-and-classify baseline that predicts the degree directly.

**Config (identical to the original run):** 100 epochs · imgsz 640 · batch 8 · AdamW lr 1e-3 · seed 42 · patience 20.
**Data:** `Burn-Benchmark2/dataset/burn-yolo-seg-clean.zip` (on Drive) · **Outputs →** `Burn-Benchmark2/models` + `results`.

In [23]:
# ===== SECTION 1 · SETUP =====
# Mount Drive (all outputs persist there), install Ultralytics, confirm the GPU.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys, torch
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics==8.3.82"], check=False)

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to H100")

WORK = "/content/drive/MyDrive/Burn-Benchmark2"
print("workspace:", WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch 2.11.0+cu128 | CUDA: True
GPU: NVIDIA A100-SXM4-40GB
workspace: /content/drive/MyDrive/Burn-Benchmark2


## Section 2 · Dataset
Unzip the leak-free `burn-yolo-seg-clean` set from Drive to fast local disk and write each `data.yaml`.

In [ ]:
# ===== SECTION 2 · DATASET =====
import os, zipfile, glob
SRC_ZIP = "/content/drive/MyDrive/Burn-Benchmark2/dataset/burn-yolo-seg-clean.zip"
DATA = "/content/burn_yolo_seg"                      # local fast working copy
if not os.path.isdir(DATA):
    with zipfile.ZipFile(SRC_ZIP) as z:
        z.extractall(DATA)
print("extracted ->", sorted(os.listdir(DATA)))

for variant, nc, names in [("1class", 1, ["Burn"]),
                           ("3class", 3, ["Degree1", "Degree2", "Degree3"])]:
    root = os.path.join(DATA, variant)
    with open(os.path.join(root, "data.yaml"), "w") as f:
        f.write(f"path: {root}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
                f"nc: {nc}\nnames: {names}\n")
    counts = {sp: len(glob.glob(f"{root}/{sp}/images/*")) for sp in ["train", "valid", "test"]}
    print(f"  {variant}: {counts}")

extracted -> ['1class', '3class']
  1class: {'train': 2425, 'valid': 206, 'test': 205}
  3class: {'train': 2425, 'valid': 206, 'test': 205}


## Section 3 · Model 1 — 1-class localiser
The segmentation stage of the deployed pipeline. Trains YOLOv8x-seg to find the burn region, saves `best.pt` to Drive.

In [ ]:
# ===== SECTION 3 · MODEL 1 — 1-class localiser =====
import os, shutil
from ultralytics import YOLO
DATA = "/content/burn_yolo_seg"; WORK = "/content/drive/MyDrive/Burn-Benchmark2"
OUT  = f"{WORK}/models/yolov8x-seg_1class_best.pt"

if os.path.exists(OUT):
    print("Model 1 already trained — skipping ->", OUT)
else:
    m1 = YOLO("yolov8x-seg.pt")
    m1.train(data=f"{DATA}/1class/data.yaml", epochs=100, imgsz=640, batch=8,
             optimizer="AdamW", lr0=1e-3, seed=42, patience=20, workers=2,
             project="/content/runs", name="1class", exist_ok=True, verbose=True)
    os.makedirs(f"{WORK}/models", exist_ok=True)
    shutil.copy("/content/runs/1class/weights/best.pt", OUT)
    print("DONE — Model 1 saved ->", OUT)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 137M/137M [00:00<00:00, 550MB/s]


New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=segment, mode=train, model=yolov8x-seg.pt, data=/content/burn_yolo_seg/1class/data.yaml, epochs=100, time=None, patience=20, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=/content/runs, name=1class, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=

100%|██████████| 755k/755k [00:00<00:00, 169MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      2320  ultralytics.nn.modules.conv.Conv             [3, 80, 3, 2]                 
  1                  -1  1    115520  ultralytics.nn.modules.conv.Conv             [80, 160, 3, 2]               
  2                  -1  3    436800  ultralytics.nn.modules.block.C2f             [160, 160, 3, True]           
  3                  -1  1    461440  ultralytics.nn.modules.conv.Conv             [160, 320, 3, 2]              
  4                  -1  6   3281920  ultralytics.nn.modules.block.C2f             [320, 320, 6, True]           
  5                  -1  1   1844480  ultralytics.nn.modules.conv.Conv             [320, 640, 3, 2]              
  6                  -1  6  13117440  ultralytics.nn.modules.block.C2f             [640, 640, 6, True]           
  7                  -1  1   3687680  ultralytics

...

     93/100         8G     0.7428       1.71     0.5954      1.264          1        640: 100%|██████████| 304/304 [00:47<00:00,  6.45it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:01<00:00,  6.63it/s]

                   all        206        299      0.743      0.707      0.741      0.461      0.802      0.637      0.716      0.414
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 73, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



93 epochs completed in 1.322 hours.
Optimizer stripped from /content/runs/1class/weights/last.pt, 143.9MB
Optimizer stripped from /content/runs/1class/weights/best.pt, 143.9MB

Validating /content/runs/1class/weights/best.pt...
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-seg summary (fused): 125 layers, 71,721,619 parameters, 0 gradients, 327.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  5.06it/s]


                   all        206        299      0.738      0.652      0.742      0.465       0.74      0.622      0.712      0.421
Speed: 0.2ms preprocess, 3.5ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/1class
DONE — Model 1 saved -> /content/drive/MyDrive/Burn-Benchmark2/models/yolov8x-seg_1class_best.pt


## Section 4 · Model 2 — 3-class standalone
The segment-and-classify baseline that predicts the burn degree directly (the pipeline's competitor).

In [ ]:
# ===== SECTION 4 · MODEL 2 — 3-class standalone =====
import os, shutil
from ultralytics import YOLO
DATA = "/content/burn_yolo_seg"; WORK = "/content/drive/MyDrive/Burn-Benchmark2"
OUT  = f"{WORK}/models/yolov8x-seg_3class_best.pt"

if os.path.exists(OUT):
    print("Model 2 already trained — skipping ->", OUT)
else:
    m3 = YOLO("yolov8x-seg.pt")
    m3.train(data=f"{DATA}/3class/data.yaml", epochs=100, imgsz=640, batch=8,
             optimizer="AdamW", lr0=1e-3, seed=42, patience=20, workers=2,
             project="/content/runs", name="3class", exist_ok=True, verbose=True)
    shutil.copy("/content/runs/3class/weights/best.pt", OUT)
    print("DONE — Model 2 saved ->", OUT)

New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=segment, mode=train, model=yolov8x-seg.pt, data=/content/burn_yolo_seg/3class/data.yaml, epochs=100, time=None, patience=20, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=/content/runs, name=3class, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=

train: Scanning /content/burn_yolo_seg/3class/train/labels... 2425 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2425/2425 [00:01<00:00, 1367.31it/s]

train: New cache created: /content/burn_yolo_seg/3class/train/labels.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /content/burn_yolo_seg/3class/valid/labels... 206 images, 0 backgrounds, 0 corrupt: 100%|██████████| 206/206 [00:00<00:00, 1191.32it/s]

...

    100/100      9.38G     0.6585      1.564      0.591       1.18          1        640: 100%|██████████| 304/304 [00:47<00:00,  6.41it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:01<00:00,  6.60it/s]

                   all        206        299       0.68      0.536      0.605      0.405      0.618       0.55       0.58      0.358



100 epochs completed in 1.426 hours.
Optimizer stripped from /content/runs/3class/weights/last.pt, 143.9MB
Optimizer stripped from /content/runs/3class/weights/best.pt, 143.9MB

Validating /content/runs/3class/weights/best.pt...
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-seg summary (fused): 125 layers, 71,723,545 parameters, 0 gradients, 328.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  5.00it/s]


                   all        206        299      0.679      0.544      0.615      0.416      0.656      0.542      0.595      0.352
               Degree1         87        136      0.716      0.529      0.626      0.376      0.695      0.537      0.605      0.322
               Degree2         64         93      0.515       0.43      0.465      0.279      0.488      0.431      0.442       0.24
               Degree3         56         70      0.806      0.671      0.753      0.592      0.785      0.657      0.739      0.495
Speed: 0.2ms preprocess, 3.7ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/3class
DONE — Model 2 saved -> /content/drive/MyDrive/Burn-Benchmark2/models/yolov8x-seg_3class_best.pt


## Section 5 · Evaluate + results
Mask mAP (val/test) for both models, plus the 3-class standalone test accuracy for the head-to-head against the pipeline. Saves `benchmark2_results.json` to Drive.

In [ ]:
# ===== SECTION 5 · EVALUATE + RESULTS =====
import os, json, glob
from collections import Counter
from ultralytics import YOLO
DATA = "/content/burn_yolo_seg"; WORK = "/content/drive/MyDrive/Burn-Benchmark2"
results = {}

# mask mAP for both models (val + shared test set)
for variant in ["1class", "3class"]:
    m = YOLO(f"{WORK}/models/yolov8x-seg_{variant}_best.pt")
    val  = m.val(data=f"{DATA}/{variant}/data.yaml", split="val",  verbose=False)
    test = m.val(data=f"{DATA}/{variant}/data.yaml", split="test", verbose=False)
    results[variant] = {"val_mask_map50": float(val.seg.map50),  "val_mask_map": float(val.seg.map),
                        "test_mask_map50": float(test.seg.map50), "test_mask_map": float(test.seg.map)}
    print(variant, results[variant])

# 3-class standalone classification accuracy on the shared 205-image test set
def true_deg(lab):
    cls = [int(l.split()[0]) for l in open(lab) if l.split()]
    return Counter(cls).most_common(1)[0][0] if cls else None
m3 = YOLO(f"{WORK}/models/yolov8x-seg_3class_best.pt")
tdir = f"{DATA}/3class/test"; correct = tot = nodet = 0
for img in glob.glob(f"{tdir}/images/*"):
    base = os.path.splitext(os.path.basename(img))[0]
    lab = f"{tdir}/labels/{base}.txt"
    if not os.path.exists(lab): continue
    td = true_deg(lab)
    if td is None: continue
    r = m3(img, imgsz=640, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        pred, nodet = None, nodet + 1
    else:
        confs = r.boxes.conf.cpu().numpy(); pred = int(r.boxes.cls.cpu().numpy()[confs.argmax()])
    correct += int(pred == td); tot += 1
acc = 100 * correct / max(1, tot)
results["3class_standalone_test_accuracy"] = {"acc": acc, "correct": correct, "n": tot, "no_detection": nodet}
print(f"\n3-class standalone test accuracy: {acc:.1f}%  ({correct}/{tot}, {nodet} no-detection)")

os.makedirs(f"{WORK}/results", exist_ok=True)
with open(f"{WORK}/results/benchmark2_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved -> Burn-Benchmark2/results/benchmark2_results.json")
print(json.dumps(results, indent=2))

Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-seg summary (fused): 125 layers, 71,721,619 parameters, 0 gradients, 327.9 GFLOPs


val: Scanning /content/burn_yolo_seg/1class/valid/labels.cache... 206 images, 0 backgrounds, 0 corrupt: 100%|██████████| 206/206 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  4.26it/s]


                   all        206        299      0.738      0.652      0.742      0.465      0.743      0.622      0.712       0.42
Speed: 0.7ms preprocess, 5.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/segment/val
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)


val: Scanning /content/burn_yolo_seg/1class/test/labels... 205 images, 0 backgrounds, 0 corrupt: 100%|██████████| 205/205 [00:00<00:00, 1314.00it/s]

val: New cache created: /content/burn_yolo_seg/1class/test/labels.cache


...

                   all        206        299      0.684      0.544      0.615      0.415      0.656      0.542      0.595      0.352
Speed: 0.9ms preprocess, 5.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/segment/val3
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)


val: Scanning /content/burn_yolo_seg/3class/test/labels... 205 images, 0 backgrounds, 0 corrupt: 100%|██████████| 205/205 [00:00<00:00, 1317.38it/s]

val: New cache created: /content/burn_yolo_seg/3class/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  4.03it/s]


                   all        205        294      0.687      0.545      0.633      0.397      0.676      0.538      0.603      0.349
Speed: 0.8ms preprocess, 5.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/segment/val4
3class {'val_mask_map50': 0.5948190138260219, 'val_mask_map': 0.3521864559664328, 'test_mask_map50': 0.602845963933915, 'test_mask_map': 0.34942444995642147}

3-class standalone test accuracy: 78.5%  (161/205, 10 no-detection)

saved -> Burn-Benchmark2/results/benchmark2_results.json
{
  "1class": {
    "val_mask_map50": 0.711938496077539,
    "val_mask_map": 0.4199379289867088,
    "test_mask_map50": 0.7262540929551761,
    "test_mask_map": 0.41694065620019305
  },
  "3class": {
    "val_mask_map50": 0.5948190138260219,
    "val_mask_map": 0.3521864559664328,
    "test_mask_map50": 0.602845963933915,
    "test_mask_map": 0.34942444995642147
  },
  "3class_standalone_test_accuracy": {
    "acc": 78.53658536585365,
    "correct": 161,
    "n

---
# Part B · Complete the honest 3-model head-to-head
Retrain + **save** the two best classifiers (ConvNeXt-Large unmasked, Swin-Tiny masked), compute the **true** end-to-end pipeline (real localiser → predicted mask → Swin, *not* oracle masks), and generate all figures to `Burn-Benchmark2/figures/`.
*(Section 6 below sets up the classifier data + shared training helper.)*

In [ ]:
# ===== SECTION 6 · CLASSIFIER DATA + SHARED HELPERS =====
import subprocess, sys, os, glob, zipfile
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm==1.0.11", "seaborn", "scikit-learn"], check=False)
import torch, torch.nn as nn, numpy as np, timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
DEV = "cuda" if torch.cuda.is_available() else "cpu"
WORK = "/content/drive/MyDrive/Burn-Benchmark2"; os.makedirs(f"{WORK}/figures", exist_ok=True)
IMAGENET = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

CLS = "/content/burn_4cond"
if not os.path.isdir(CLS):
    with zipfile.ZipFile(f"{WORK}/dataset/burn-clean-4cond.zip") as z: z.extractall(CLS)
CLS_ROOT = CLS
for r, ds, fs in os.walk(CLS):
    if os.path.isdir(os.path.join(r, "unmasked")) and os.path.isdir(os.path.join(r, "masked")):
        CLS_ROOT = r; break
print("classifier root:", CLS_ROOT, sorted(os.listdir(CLS_ROOT)))

def tfm(train):
    ops = [transforms.Resize((224, 224))]
    if train: ops += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2, 0.2, 0.2), transforms.RandomRotation(10)]
    return transforms.Compose(ops + [transforms.ToTensor(), transforms.Normalize(*IMAGENET)])

def train_classifier(arch_id, cond, seed=42, epochs=30):
    torch.manual_seed(seed); np.random.seed(seed)
    root = f"{CLS_ROOT}/{cond}"
    tr = datasets.ImageFolder(f"{root}/train", tfm(True)); va = datasets.ImageFolder(f"{root}/valid", tfm(False)); te = datasets.ImageFolder(f"{root}/test", tfm(False))
    cnt = np.bincount([y for _, y in tr.samples], minlength=3).astype(float)
    w = torch.tensor(cnt.sum() / (3 * np.clip(cnt, 1, None)), dtype=torch.float32, device=DEV)
    dltr = DataLoader(tr, 16, shuffle=True, num_workers=2, drop_last=True); dlva = DataLoader(va, 32, num_workers=2); dlte = DataLoader(te, 32, num_workers=2)
    model = timm.create_model(arch_id, pretrained=True, num_classes=3).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(weight=w); scaler = torch.cuda.amp.GradScaler(); best_va = -1; best = None; bad = 0
    for ep in range(epochs):
        model.train()
        for x, y in dltr:
            x, y = x.to(DEV), y.to(DEV); opt.zero_grad()
            with torch.cuda.amp.autocast(): loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step(); model.eval(); c = t = 0
        with torch.no_grad():
            for x, y in dlva: c += (model(x.to(DEV)).argmax(1).cpu() == y).sum().item(); t += len(y)
        if c / t > best_va: best_va = c / t; best = {k: v.cpu().clone() for k, v in model.state_dict().items()}; bad = 0
        else:
            bad += 1
            if bad >= 5: break
    model.load_state_dict(best); model.eval(); ys = []; ps = []
    with torch.no_grad():
        for x, y in dlte: ps += model(x.to(DEV)).argmax(1).cpu().tolist(); ys += y.tolist()
    acc = 100 * float(np.mean(np.array(ys) == np.array(ps)))
    return model, ys, ps, acc, te.classes
print("helpers ready · device:", DEV)

classifier root: /content/burn_4cond ['cropped', 'masked', 'maskedcrop', 'unmasked']
helpers ready · device: cuda


In [ ]:
# ===== SECTION 7 · RETRAIN + SAVE THE TWO BEST CLASSIFIERS =====
import torch, os, json, numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import confusion_matrix
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
LBL = ["1st", "2nd", "3rd"]

def eval_saved(arch, cond, path):
    m = timm.create_model(arch, pretrained=False, num_classes=3)
    m.load_state_dict(torch.load(path, map_location=DEV)); m = m.to(DEV).eval()
    te = datasets.ImageFolder(f"{CLS_ROOT}/{cond}/test", tfm(False)); dl = DataLoader(te, 32, num_workers=2)
    ys, ps = [], []
    with torch.no_grad():
        for x, y in dl: ps += m(x.to(DEV)).argmax(1).cpu().tolist(); ys += y.tolist()
    return ys, ps, 100 * float(np.mean(np.array(ys) == np.array(ps)))

CFG = [("convnext_large", "unmasked", "convnext_large_unmasked", "Standalone classifier"),
       ("swin_tiny_patch4_window7_224", "masked", "swin_tiny_masked", "Pipeline classifier")]
cls_results = {}
for arch, cond, tag, label in CFG:
    out = f"{WORK}/models/{tag}.pth"
    if os.path.exists(out):
        ys, ps, acc = eval_saved(arch, cond, out); print(f"{label}: loaded {tag} — TEST {acc:.1f}%")
    else:
        print(f"training {label} ({arch} / {cond}) ...", flush=True)
        model, ys, ps, acc, _ = train_classifier(arch, cond)
        torch.save(model.state_dict(), out); print(f"{label}: trained {tag} — TEST {acc:.1f}% -> saved")
    cm = confusion_matrix(ys, ps, labels=[0, 1, 2])
    cls_results[tag] = {"arch": arch, "cond": cond, "test_acc": acc, "cm": cm.tolist()}
    plt.figure(figsize=(3.8, 3.2)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LBL, yticklabels=LBL)
    plt.title(f"{label}\n{arch} ({cond}) — {acc:.1f}%"); plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
    plt.savefig(f"{WORK}/figures/cm_{tag}.png", dpi=150); plt.close()
os.makedirs(f"{WORK}/results", exist_ok=True)
json.dump(cls_results, open(f"{WORK}/results/classifier_results.json", "w"), indent=2)
print("\nclassifier results:", {k: round(v['test_acc'], 1) for k, v in cls_results.items()})

training Standalone classifier (convnext_large / unmasked) ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  791MB            

model.safetensors: downloading bytes:           |  0.00B            

Standalone classifier: trained convnext_large_unmasked — TEST 82.4% -> saved
training Pipeline classifier (swin_tiny_patch4_window7_224 / masked) ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Pipeline classifier: trained swin_tiny_masked — TEST 83.9% -> saved

classifier results: {'convnext_large_unmasked': 82.4, 'swin_tiny_masked': 83.9}


In [ ]:
# ===== SECTION 8 · TRUE END-TO-END PIPELINE (real localiser -> mask -> Swin) =====
import cv2, glob, os, json, numpy as np, torch, timm
from PIL import Image
from ultralytics import YOLO
from collections import Counter
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
LBL = ["1st", "2nd", "3rd"]

yolo = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")            # the real localiser
swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=3)
swin.load_state_dict(torch.load(f"{WORK}/models/swin_tiny_masked.pth", map_location=DEV)); swin = swin.to(DEV).eval()

def true_deg(lab):
    c = [int(l.split()[0]) for l in open(lab) if l.split()]
    return Counter(c).most_common(1)[0][0] if c else None

def pipe_pred(img_path):
    bgr = cv2.imread(img_path); r = yolo(bgr, imgsz=640, verbose=False)[0]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is None or len(r.boxes) == 0: return None            # localiser found nothing
    conf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(np.argmax(conf))].cpu().numpy()
    mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); masked = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    x = tfm(False)(Image.fromarray(masked.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(swin(x).argmax(1))

tdir = "/content/burn_yolo_seg/3class/test"; ys, ps, nodet = [], [], 0
for img in glob.glob(f"{tdir}/images/*"):
    base = os.path.splitext(os.path.basename(img))[0]; lab = f"{tdir}/labels/{base}.txt"
    if not os.path.exists(lab): continue
    td = true_deg(lab)
    if td is None: continue
    pr = pipe_pred(img)
    if pr is None: nodet += 1; pr = -1                              # no-detection = error (kept in denominator)
    ys.append(td); ps.append(pr)
ys, ps = np.array(ys), np.array(ps); n = len(ys); correct = int((ys == ps).sum()); acc = 100 * correct / n
print(f"TRUE PIPELINE (localiser -> mask -> Swin-Tiny) TEST acc: {acc:.1f}%  ({correct}/{n}, {nodet} no-detection = errors)")
cm = confusion_matrix(ys, np.where(ps < 0, 3, ps), labels=[0, 1, 2, 3])[:3]
plt.figure(figsize=(4.4, 3.2)); sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=LBL + ["no-det"], yticklabels=LBL)
plt.title(f"True pipeline (real masks) — {acc:.1f}%"); plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
plt.savefig(f"{WORK}/figures/cm_true_pipeline.png", dpi=150); plt.close()
json.dump({"true_pipeline_test_acc": acc, "n": n, "correct": correct, "no_detection": nodet},
          open(f"{WORK}/results/true_pipeline_results.json", "w"), indent=2)
print("saved -> true_pipeline_results.json + confusion matrix")

TRUE PIPELINE (localiser -> mask -> Swin-Tiny) TEST acc: 79.0%  (162/205, 10 no-detection = errors)
saved -> true_pipeline_results.json + confusion matrix


In [ ]:
# ===== SECTION 9 · HEAD-TO-HEAD CHART + YOLO FIGURES =====
import json, os, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
cls  = json.load(open(f"{WORK}/results/classifier_results.json"))
pipe = json.load(open(f"{WORK}/results/true_pipeline_results.json"))
b2   = json.load(open(f"{WORK}/results/benchmark2_results.json"))
N = pipe["n"]

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d
    h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return 100*(c-h), 100*(c+h)

rows = [("Standalone\nYOLOv8x-seg", b2["3class_standalone_test_accuracy"]["acc"], b2["3class_standalone_test_accuracy"]["correct"]),
        ("Pipeline\nloc->Swin",     pipe["true_pipeline_test_acc"], pipe["correct"]),
        ("Standalone\nConvNeXt-L",  cls["convnext_large_unmasked"]["test_acc"], round(cls["convnext_large_unmasked"]["test_acc"]/100*N))]
names = [r[0] for r in rows]; vals = [r[1] for r in rows]
lo = [wilson(r[2], N)[0] for r in rows]; hi = [wilson(r[2], N)[1] for r in rows]
err = [[v-l for v, l in zip(vals, lo)], [h-v for v, h in zip(vals, hi)]]
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(names, vals, color=["#d95f02", "#1b9e77", "#7570b3"], yerr=err, capsize=6)
for i, v in enumerate(vals): ax.text(i, v + 1.4, f"{v:.1f}%", ha='center', fontweight='bold')
ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(55, 95)
ax.set_title(f"Burn-degree head-to-head — leak-free test (N={N}), 95% Wilson CI")
plt.tight_layout(); plt.savefig(f"{WORK}/figures/head_to_head.png", dpi=160); plt.close()
print("head-to-head chart saved. numbers:", [(n_, round(v,1)) for n_, v in zip(names, vals)])

# regenerate YOLO confusion matrices + PR curves from the saved best.pt
for variant in ["1class", "3class"]:
    YOLO(f"{WORK}/models/yolov8x-seg_{variant}_best.pt").val(
        data=f"/content/burn_yolo_seg/{variant}/data.yaml", split="test", plots=True,
        project=f"{WORK}/figures", name=f"yolo_{variant}_test", exist_ok=True, verbose=False)
print("\nALL FIGURES in Burn-Benchmark2/figures/:", sorted(os.listdir(f"{WORK}/figures")))

head-to-head chart saved. numbers: [('Standalone\nYOLOv8x-seg', 78.5), ('Pipeline\nloc->Swin', 79.0), ('Standalone\nConvNeXt-L', 82.4)]
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-seg summary (fused): 125 layers, 71,721,619 parameters, 0 gradients, 327.9 GFLOPs


val: Scanning /content/burn_yolo_seg/1class/test/labels.cache... 205 images, 0 backgrounds, 0 corrupt: 100%|██████████| 205/205 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  4.24it/s]


                   all        205        294      0.789      0.663      0.765      0.469      0.751      0.663      0.726      0.417
Speed: 0.8ms preprocess, 5.2ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/drive/MyDrive/Burn-Benchmark2/figures/yolo_1class_test
Ultralytics 8.3.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8x-seg summary (fused): 125 layers, 71,723,545 parameters, 0 gradients, 328.0 GFLOPs


val: Scanning /content/burn_yolo_seg/3class/test/labels.cache... 205 images, 0 backgrounds, 0 corrupt: 100%|██████████| 205/205 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:03<00:00,  4.03it/s]


                   all        205        294      0.687      0.545      0.633      0.397      0.676      0.538      0.603      0.349
Speed: 0.7ms preprocess, 5.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/drive/MyDrive/Burn-Benchmark2/figures/yolo_3class_test

ALL FIGURES in Burn-Benchmark2/figures/: ['cm_convnext_large_unmasked.png', 'cm_swin_tiny_masked.png', 'cm_true_pipeline.png', 'head_to_head.png', 'yolo_1class_test', 'yolo_3class_test']


---
## Section 10 · External generalization test (clean 319-image set)
Runs all three models on the leak-verified external set (independent sources, 226 BIAC-duplicate images removed). Reports accuracy + **balanced** accuracy (the set is imbalanced) + confusion matrices → a second generalization data point alongside BIP_US. Inference only, ~3–5 min.

In [ ]:
# ===== SECTION 10 · EXTERNAL GENERALIZATION EVAL (clean 319-image set) =====
import os, glob, json, zipfile, numpy as np, cv2, torch, timm
from PIL import Image
from ultralytics import YOLO
from collections import Counter
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt, seaborn as sns
WORK = "/content/drive/MyDrive/Burn-Benchmark2"; LBL = ["1st", "2nd", "3rd"]

EXT = "/content/external"
if not os.path.isdir(EXT):
    with zipfile.ZipFile(f"{WORK}/dataset/burn-external-test-clean.zip") as z: z.extractall(EXT)
IMG = None
for r, ds, fs in os.walk(EXT):
    if os.path.isdir(os.path.join(r, "images")) and os.path.isdir(os.path.join(r, "labels")): IMG = r; break
print("external root:", IMG, "| images:", len(glob.glob(f"{IMG}/images/*")))

yolo3 = YOLO(f"{WORK}/models/yolov8x-seg_3class_best.pt")
yolo1 = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")
swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=3)
swin.load_state_dict(torch.load(f"{WORK}/models/swin_tiny_masked.pth", map_location=DEV)); swin = swin.to(DEV).eval()
cnx = timm.create_model("convnext_large", pretrained=False, num_classes=3)
cnx.load_state_dict(torch.load(f"{WORK}/models/convnext_large_unmasked.pth", map_location=DEV)); cnx = cnx.to(DEV).eval()

def true_deg(lab):
    c = [int(l.split()[0]) for l in open(lab) if l.split()]; return Counter(c).most_common(1)[0][0] if c else None
def yolo_pred(m, bgr):
    r = m(bgr, imgsz=640, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0: return -1
    cf = r.boxes.conf.cpu().numpy(); return int(r.boxes.cls.cpu().numpy()[cf.argmax()])
def pipe_pred(bgr):
    r = yolo1(bgr, imgsz=640, verbose=False)[0]; rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is None or len(r.boxes) == 0: return -1
    cf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(cf.argmax())].cpu().numpy()
    mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); masked = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    x = tfm(False)(Image.fromarray(masked.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(swin(x).argmax(1))
def cnx_pred(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); x = tfm(False)(Image.fromarray(rgb)).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(cnx(x).argmax(1))

yt = []; P = {"yolo": [], "pipe": [], "cnx": []}
for img in glob.glob(f"{IMG}/images/*"):
    base = os.path.splitext(os.path.basename(img))[0]; lab = f"{IMG}/labels/{base}.txt"
    if not os.path.exists(lab): continue
    td = true_deg(lab)
    if td is None: continue
    bgr = cv2.imread(img)
    if bgr is None: continue
    yt.append(td)
    P["yolo"].append(yolo_pred(yolo3, bgr)); P["pipe"].append(pipe_pred(bgr)); P["cnx"].append(cnx_pred(bgr))
yt = np.array(yt); N = len(yt)
ext = {}
for key, name, cmap in [("cnx", "Standalone ConvNeXt-L", "Oranges"), ("pipe", "Pipeline loc->Swin", "Oranges"), ("yolo", "Standalone YOLOv8-seg", "Oranges")]:
    pp = np.array(P[key]); acc = 100 * float((pp == yt).mean())
    bal = 100 * balanced_accuracy_score(yt, np.where(pp < 0, 99, pp))
    ext[key] = {"name": name, "acc": acc, "balanced_acc": bal, "n": int(N), "correct": int((pp == yt).sum()),
                "no_detection": int((pp < 0).sum())}
    cm = confusion_matrix(yt, np.where(pp < 0, 3, pp), labels=[0, 1, 2, 3])[:3]
    plt.figure(figsize=(4.4, 3.2)); sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=LBL + ["no-det"], yticklabels=LBL)
    plt.title(f"EXTERNAL: {name} — {acc:.1f}% (bal {bal:.1f}%)"); plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
    plt.savefig(f"{WORK}/figures/ext_cm_{key}.png", dpi=150); plt.close()
    print(f"EXTERNAL {name}: acc {acc:.1f}%  | balanced {bal:.1f}%  | {int((pp<0).sum())} no-detection")
json.dump({"N": int(N), "per_degree": {LBL[k]: int((yt == k).sum()) for k in [0, 1, 2]}, "models": ext},
          open(f"{WORK}/results/external_eval_results.json", "w"), indent=2)
print("\nsaved -> external_eval_results.json + external confusion matrices. N =", N,
      "| degree dist:", {LBL[k]: int((yt == k).sum()) for k in [0, 1, 2]})

external root: /content/external | images: 319
EXTERNAL Standalone ConvNeXt-L: acc 74.3%  | balanced 78.7%  | 0 no-detection
EXTERNAL Pipeline loc->Swin: acc 54.2%  | balanced 54.7%  | 66 no-detection


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


EXTERNAL Standalone YOLOv8-seg: acc 58.0%  | balanced 65.5%  | 55 no-detection

saved -> external_eval_results.json + external confusion matrices. N = 319 | degree dist: {'1st': 139, '2nd': 157, '3rd': 23}


---
## Section 11 · Robust pipeline (fair-chance: low threshold + full-frame fallback)
Gives the pipeline its best honest shot, matching the deployed app: lower the localiser confidence threshold, and if it *still* detects nothing, fall back to classifying the **full image** with Swin (never auto-fail). Evaluated on **both** internal (205) and external (319). Reported alongside the strict pipeline for full transparency (no bias). Inference only.

In [ ]:
# ===== SECTION 11 · ROBUST PIPELINE (low localiser threshold + full-frame fallback) =====
import os, glob, json, numpy as np, cv2, torch, timm
from PIL import Image
from ultralytics import YOLO
from collections import Counter
from sklearn.metrics import balanced_accuracy_score
WORK = "/content/drive/MyDrive/Burn-Benchmark2"; LBL = ["1st", "2nd", "3rd"]
yolo1 = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")
swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=3)
swin.load_state_dict(torch.load(f"{WORK}/models/swin_tiny_masked.pth", map_location=DEV)); swin = swin.to(DEV).eval()

def classify_full(rgb):
    x = tfm(False)(Image.fromarray(rgb.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(swin(x).argmax(1))

def robust_pipe(bgr, conf=0.05):
    r = yolo1(bgr, imgsz=640, conf=conf, verbose=False)[0]; rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is None or len(r.boxes) == 0:
        return classify_full(rgb), True                       # fallback: full image -> Swin
    cf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(cf.argmax())].cpu().numpy()
    mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); masked = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    return classify_full(masked), False

def true_deg(lab):
    c = [int(l.split()[0]) for l in open(lab) if l.split()]; return Counter(c).most_common(1)[0][0] if c else None

def eval_set(img_dir, lab_dir):
    yt, pp, fb = [], [], 0
    for img in glob.glob(f"{img_dir}/*"):
        base = os.path.splitext(os.path.basename(img))[0]; lab = f"{lab_dir}/{base}.txt"
        if not os.path.exists(lab): continue
        td = true_deg(lab)
        if td is None: continue
        bgr = cv2.imread(img)
        if bgr is None: continue
        p, used = robust_pipe(bgr); yt.append(td); pp.append(p); fb += int(used)
    yt, pp = np.array(yt), np.array(pp); n = len(yt)
    return 100 * float((pp == yt).mean()), 100 * balanced_accuracy_score(yt, pp), n, fb

res = {}
sets = {"internal": ("/content/burn_yolo_seg/3class/test/images", "/content/burn_yolo_seg/3class/test/labels"),
        "external": ("/content/external/images", "/content/external/labels")}
for tag, (idir, ldir) in sets.items():
    acc, bal, n, fb = eval_set(idir, ldir)
    res[tag] = {"robust_pipeline_acc": acc, "balanced_acc": bal, "n": n, "fallback_used": fb}
    print(f"ROBUST PIPELINE {tag}: acc {acc:.1f}%  | balanced {bal:.1f}%  (N={n}, fallback used on {fb} imgs)")
json.dump(res, open(f"{WORK}/results/robust_pipeline_results.json", "w"), indent=2)
print("\nsaved -> robust_pipeline_results.json")

ROBUST PIPELINE internal: acc 81.5%  | balanced 80.9%  (N=205, fallback used on 0 imgs)
ROBUST PIPELINE external: acc 69.9%  | balanced 66.3%  (N=319, fallback used on 1 imgs)

saved -> robust_pipeline_results.json


---
## Section 12 · Multi-seed robustness (3 seeds) — the key comparison
Retrains ConvNeXt-Large (standalone) + Swin-Tiny (pipeline classifier) at seeds 0/1/2, re-evaluates the **standalone classifier** and the **robust pipeline** (localiser fixed) on internal (205) + external (319), and reports **mean ± SD**. This turns "single run, CIs overlap" into a statistically credible comparison. Longest run (~1 h) — keep the tab in front.

In [ ]:
# ===== SECTION 12 · MULTI-SEED ROBUSTNESS (3 seeds: classifier vs robust pipeline) =====
# Resumable: saves after each seed, skips seeds already done on re-run.
import os, json, glob, numpy as np, cv2, torch
from PIL import Image
from ultralytics import YOLO
from collections import Counter, defaultdict
from sklearn.metrics import balanced_accuracy_score
WORK = "/content/drive/MyDrive/Burn-Benchmark2"
yolo1 = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")   # localiser fixed across seeds
RESF = f"{WORK}/results/multiseed_results.json"

def true_deg(lab):
    c = [int(l.split()[0]) for l in open(lab) if l.split()]; return Counter(c).most_common(1)[0][0] if c else None
def pairs(idir, ldir):
    out = []
    for img in glob.glob(f"{idir}/*"):
        b = os.path.splitext(os.path.basename(img))[0]; lab = f"{ldir}/{b}.txt"
        if os.path.exists(lab):
            td = true_deg(lab)
            if td is not None: out.append((img, td))
    return out
INT = pairs("/content/burn_yolo_seg/3class/test/images", "/content/burn_yolo_seg/3class/test/labels")
EXT = pairs("/content/external/images", "/content/external/labels")
IMGS = {f: cv2.imread(f) for f, _ in INT + EXT}

def clf_pred(model, bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); x = tfm(False)(Image.fromarray(rgb)).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(model(x).argmax(1))
def pipe_pred(swin, bgr, conf=0.05):
    r = yolo1(bgr, imgsz=640, conf=conf, verbose=False)[0]; rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is None or len(r.boxes) == 0:
        img = rgb
    else:
        cf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(cf.argmax())].cpu().numpy()
        mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); img = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    x = tfm(False)(Image.fromarray(img.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(swin(x).argmax(1))
def score(ps, predfn):
    yt = np.array([t for _, t in ps]); pp = np.array([predfn(IMGS[f]) for f, _ in ps])
    return 100 * float((pp == yt).mean()), 100 * balanced_accuracy_score(yt, pp)

acc = defaultdict(list); done = []
if os.path.exists(RESF):
    prev = json.load(open(RESF))
    if "_raw" in prev:
        acc = defaultdict(list, {k: list(v) for k, v in prev["_raw"].items()}); done = prev.get("_seeds_done", [])
        print("resuming; seeds already done:", done)

for seed in [0, 1, 2]:
    if seed in done: print(f"seed {seed} done — skip"); continue
    print(f"seed {seed}: training classifiers ...", flush=True)
    cnx, *_ = train_classifier("convnext_large", "unmasked", seed=seed)
    swin, *_ = train_classifier("swin_tiny_patch4_window7_224", "masked", seed=seed)
    cnx.eval(); swin.eval()
    for tag, ps in [("int", INT), ("ext", EXT)]:
        a, b = score(ps, lambda bgr: clf_pred(cnx, bgr));   acc[f"clf_{tag}"].append(a);  acc[f"clf_{tag}_bal"].append(b)
        a, b = score(ps, lambda bgr: pipe_pred(swin, bgr)); acc[f"pipe_{tag}"].append(a); acc[f"pipe_{tag}_bal"].append(b)
    done.append(seed)
    json.dump({"_raw": dict(acc), "_seeds_done": done}, open(RESF, "w"))
    print(f"  seed {seed}: clf_int {acc['clf_int'][-1]:.1f} pipe_int {acc['pipe_int'][-1]:.1f} | "
          f"clf_ext {acc['clf_ext'][-1]:.1f} pipe_ext {acc['pipe_ext'][-1]:.1f}  [saved]", flush=True)

summ = {k: {"mean": float(np.mean(v)), "sd": float(np.std(v)), "seeds": list(v)} for k, v in acc.items()}
summ["_raw"] = dict(acc); summ["_seeds_done"] = done
json.dump(summ, open(RESF, "w"), indent=2)
print("\n=== MULTI-SEED (mean ± SD over seeds", done, ") ===")
for k in ["clf_int", "pipe_int", "clf_ext", "pipe_ext", "clf_ext_bal", "pipe_ext_bal"]:
    if k in summ: print(f"  {k:14s}: {summ[k]['mean']:.1f} ± {summ[k]['sd']:.1f}")
print("saved -> multiseed_results.json")

seed 0: training classifiers ...
  seed 0: clf_int 82.0 pipe_int 78.0 | clf_ext 76.8 pipe_ext 70.5  [saved]
seed 1: training classifiers ...
  seed 1: clf_int 82.4 pipe_int 78.5 | clf_ext 74.9 pipe_ext 74.3  [saved]
seed 2: training classifiers ...
  seed 2: clf_int 83.4 pipe_int 80.0 | clf_ext 78.1 pipe_ext 75.2  [saved]

=== MULTI-SEED (mean ± SD over seeds [0, 1, 2] ) ===
  clf_int       : 82.6 ± 0.6
  pipe_int      : 78.9 ± 0.8
  clf_ext       : 76.6 ± 1.3
  pipe_ext      : 73.4 ± 2.0
  clf_ext_bal   : 80.1 ± 1.5
  pipe_ext_bal  : 77.9 ± 2.7
saved -> multiseed_results.json


---
## Section 13 · Efficiency / timing table
Per-image inference time for each stage and the full pipeline on the A100 — a stated benchmark goal and a professional touch for the deployed-system story. Fast (~1 min).

In [ ]:
# ===== SECTION 13 · EFFICIENCY / TIMING (per-stage, A100) =====
import time, glob, json, numpy as np, cv2, torch, timm
from PIL import Image
from ultralytics import YOLO
WORK = "/content/drive/MyDrive/Burn-Benchmark2"
yolo1 = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")
yolo3 = YOLO(f"{WORK}/models/yolov8x-seg_3class_best.pt")
swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=3)
swin.load_state_dict(torch.load(f"{WORK}/models/swin_tiny_masked.pth", map_location=DEV)); swin = swin.to(DEV).eval()
cnx = timm.create_model("convnext_large", pretrained=False, num_classes=3)
cnx.load_state_dict(torch.load(f"{WORK}/models/convnext_large_unmasked.pth", map_location=DEV)); cnx = cnx.to(DEV).eval()
data = [cv2.imread(f) for f in glob.glob("/content/burn_yolo_seg/3class/test/images/*")[:60]]

def bench(fn, warm=8):
    for d in data[:warm]: fn(d)
    torch.cuda.synchronize(); t = time.time()
    for d in data: fn(d)
    torch.cuda.synchronize(); return 1000 * (time.time() - t) / len(data)
def clf_fn(model):
    def g(bgr):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); x = tfm(False)(Image.fromarray(rgb)).unsqueeze(0).to(DEV)
        with torch.no_grad(): model(x)
    return g
def pipe_fn(bgr):
    r = yolo1(bgr, imgsz=640, conf=0.05, verbose=False)[0]; rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is not None and len(r.boxes) > 0:
        cf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(cf.argmax())].cpu().numpy()
        mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); rgb = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    x = tfm(False)(Image.fromarray(rgb.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): swin(x)

T = {"YOLO localiser (1-class)": bench(lambda b: yolo1(b, imgsz=640, conf=0.05, verbose=False)),
     "Standalone YOLOv8-seg (3-class)": bench(lambda b: yolo3(b, imgsz=640, verbose=False)),
     "Swin-Tiny classifier": bench(clf_fn(swin)),
     "ConvNeXt-L classifier": bench(clf_fn(cnx)),
     "Full pipeline (loc->mask->Swin)": bench(pipe_fn)}
print("=== per-image inference time (A100, imgsz 640, batch 1) ===")
for k, v in T.items(): print(f"  {k:34s}: {v:6.1f} ms  ({1000/v:4.0f} img/s)")
json.dump(T, open(f"{WORK}/results/timing_results.json", "w"), indent=2)
print("saved -> timing_results.json")

=== per-image inference time (A100, imgsz 640, batch 1) ===
  YOLO localiser (1-class)          :   15.4 ms  (  65 img/s)
  Standalone YOLOv8-seg (3-class)   :   15.1 ms  (  66 img/s)
  Swin-Tiny classifier              :   23.7 ms  (  42 img/s)
  ConvNeXt-L classifier             :   25.8 ms  (  39 img/s)
  Full pipeline (loc->mask->Swin)   :   42.3 ms  (  24 img/s)
saved -> timing_results.json


---
## Section 14 · Paired significance (McNemar) + export proof
Paired McNemar test (classifier vs robust pipeline on the *same* images) for internal + external, saves per-image predictions, and bundles all figures + result JSONs into `benchmark2_proof.zip` on Drive for the manuscript/Zenodo.

In [ ]:
# ===== SECTION 14 · McNemar paired significance + export proof zip =====
import os, glob, json, shutil, numpy as np, cv2, torch, timm
from PIL import Image
from ultralytics import YOLO
from collections import Counter
from scipy.stats import binomtest
WORK = "/content/drive/MyDrive/Burn-Benchmark2"
yolo1 = YOLO(f"{WORK}/models/yolov8x-seg_1class_best.pt")
def load(a, f):
    m = timm.create_model(a, pretrained=False, num_classes=3); m.load_state_dict(torch.load(f"{WORK}/models/{f}", map_location=DEV)); return m.to(DEV).eval()
swin = load("swin_tiny_patch4_window7_224", "swin_tiny_masked.pth"); cnx = load("convnext_large", "convnext_large_unmasked.pth")
def true_deg(l):
    c = [int(x.split()[0]) for x in open(l) if x.split()]; return Counter(c).most_common(1)[0][0] if c else None
def clf(m, bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); x = tfm(False)(Image.fromarray(rgb)).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(m(x).argmax(1))
def pipe(bgr, conf=0.05):
    r = yolo1(bgr, imgsz=640, conf=conf, verbose=False)[0]; rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if r.masks is None or len(r.boxes) == 0: img = rgb
    else:
        cf = r.boxes.conf.cpu().numpy(); mk = r.masks.data[int(cf.argmax())].cpu().numpy(); mk = cv2.resize(mk, (rgb.shape[1], rgb.shape[0])); img = rgb * (mk > 0.1).astype(np.uint8)[:, :, None]
    x = tfm(False)(Image.fromarray(img.astype(np.uint8))).unsqueeze(0).to(DEV)
    with torch.no_grad(): return int(swin(x).argmax(1))
def mcnemar(idir, ldir, tag):
    b = c = 0; rows = []
    for im in glob.glob(f"{idir}/*"):
        base = os.path.splitext(os.path.basename(im))[0]; lab = f"{ldir}/{base}.txt"
        if not os.path.exists(lab): continue
        td = true_deg(lab)
        if td is None: continue
        bgr = cv2.imread(im)
        if bgr is None: continue
        pc = clf(cnx, bgr); pp = pipe(bgr); cc = int(pc == td); cp = int(pp == td)
        b += int(cc and not cp); c += int(cp and not cc)
        rows.append({"img": os.path.basename(im), "true": td, "clf": pc, "pipe": pp})
    p = binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) > 0 else 1.0
    json.dump(rows, open(f"{WORK}/results/preds_{tag}.json", "w"))
    print(f"McNemar {tag}: classifier-only-correct(b)={b}, pipeline-only-correct(c)={c}, discordant={b+c}, exact p={p:.4f}")
    return {"classifier_only_correct": b, "pipeline_only_correct": c, "exact_p": p, "n_discordant": b + c}
mc = {"internal": mcnemar("/content/burn_yolo_seg/3class/test/images", "/content/burn_yolo_seg/3class/test/labels", "internal"),
      "external": mcnemar("/content/external/images", "/content/external/labels", "external")}
json.dump(mc, open(f"{WORK}/results/mcnemar_results.json", "w"), indent=2)
EXP = "/content/proof"; shutil.rmtree(EXP, ignore_errors=True); os.makedirs(EXP)
shutil.copytree(f"{WORK}/figures", f"{EXP}/figures"); shutil.copytree(f"{WORK}/results", f"{EXP}/results")
shutil.make_archive(f"{WORK}/benchmark2_proof", "zip", EXP)
print("\nzipped -> Burn-Benchmark2/benchmark2_proof.zip |", os.path.getsize(f"{WORK}/benchmark2_proof.zip") // 1024, "KB")

McNemar internal: classifier-only-correct(b)=20, pipeline-only-correct(c)=18, discordant=38, exact p=0.8714
McNemar external: classifier-only-correct(b)=54, pipeline-only-correct(c)=40, discordant=94, exact p=0.1797

zipped -> Burn-Benchmark2/benchmark2_proof.zip | 6322 KB


---
## Section 15 · Fairness fix — standalone YOLO at the same low threshold
The robust pipeline used conf=0.05, but the standalone YOLOv8-seg baseline was evaluated at YOLO's default 0.25 (auto-failing when it found nothing). For a fair benchmark, re-evaluate it at **both** 0.25 and 0.05 on internal + external, so every approach gets its best honest chance.

In [ ]:
# ===== SECTION 15 · FAIR STANDALONE YOLO (low threshold, matching the pipeline) =====
import os, glob, json, numpy as np, cv2
from ultralytics import YOLO
from collections import Counter
from sklearn.metrics import balanced_accuracy_score
WORK = "/content/drive/MyDrive/Burn-Benchmark2"
yolo3 = YOLO(f"{WORK}/models/yolov8x-seg_3class_best.pt")
def true_deg(l):
    c = [int(x.split()[0]) for x in open(l) if x.split()]; return Counter(c).most_common(1)[0][0] if c else None
def ypred(bgr, conf):
    r = yolo3(bgr, imgsz=640, conf=conf, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0: return -1
    cf = r.boxes.conf.cpu().numpy(); return int(r.boxes.cls.cpu().numpy()[cf.argmax()])
def evalset(idir, ldir, conf):
    yt, pp, nd = [], [], 0
    for im in glob.glob(f"{idir}/*"):
        b = os.path.splitext(os.path.basename(im))[0]; lab = f"{ldir}/{b}.txt"
        if not os.path.exists(lab): continue
        td = true_deg(lab)
        if td is None: continue
        bgr = cv2.imread(im)
        if bgr is None: continue
        p = ypred(bgr, conf); nd += int(p < 0); yt.append(td); pp.append(p)
    yt, pp = np.array(yt), np.array(pp)
    return 100 * float((pp == yt).mean()), 100 * balanced_accuracy_score(yt, np.where(pp < 0, 99, pp)), len(yt), nd
res = {}
sets = {"internal": ("/content/burn_yolo_seg/3class/test/images", "/content/burn_yolo_seg/3class/test/labels"),
        "external": ("/content/external/images", "/content/external/labels")}
for tag, (idir, ldir) in sets.items():
    for conf in [0.25, 0.05]:
        a, b, n, nd = evalset(idir, ldir, conf)
        res[f"{tag}_conf{conf}"] = {"acc": a, "balanced": b, "n": n, "no_detection": nd}
        print(f"standalone YOLO {tag:8s} conf={conf}: acc {a:.1f}%  bal {b:.1f}%  ({nd} no-detection)")
json.dump(res, open(f"{WORK}/results/standalone_yolo_thresholds.json", "w"), indent=2)
print("saved -> standalone_yolo_thresholds.json")